# 45分で作る、はじめての予測モデル

このノートブックでは、実際の医療データを使って「糖尿病かどうかを予測するモデル」を45分で作ります。

## 目次
1. 機械学習とは？（5分）
2. データを読み込んで眺める（10分）
3. データのそうじ（10分）
4. 予測モデルを作る（10分）
5. どれくらい当たるか確かめる（5分）
6. 新しい患者さんで予測してみる（5分）

> このノートブックは1コマ目（Python_45min.ipynb）の続きです。
> セルを上から順に実行（Shift + Enter）しながら進めてください。

## 1. 機械学習とは？（5分）

**機械学習**とは、コンピュータがデータからパターンを学び、新しいデータに対して予測を行う技術です。

### 医療分野での応用例
- **診断支援**: 検査値から疾患のリスクを予測
- **予後予測**: 治療効果や再入院リスクの予測
- **画像診断**: X線やCT画像からの病変検出

### 今日作るもの
**糖尿病リスクの予測モデル**を作ります。使うのは Pima Indians Diabetes Dataset という、実際の医療研究で使われてきた768名分の健康データです。

流れはシンプルです：

```
データを読む → そうじする → モデルに学ばせる → 当たり具合を確認 → 予測に使う
```

これは規模の大小はあれど、実際のデータ分析・機械学習プロジェクトと同じ流れです。

In [ ]:
# 必要なライブラリ（道具箱）のインポート
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix

# 日本語フォントの設定（Google Colabでの文字化けを防ぐ）
!pip install japanize-matplotlib -q
import japanize_matplotlib

print("準備完了！")

## 2. データを読み込んで眺める（10分）

まずはデータを読み込んで、どんなデータなのか確認します。
分析の第一歩は、いつも「データをよく見る」ことです。

### データの内容（768名の女性の健康データ）
| 列名 | 意味 |
|---|---|
| Pregnancies | 妊娠回数 |
| Glucose | 血糖値 (mg/dL) |
| BloodPressure | 拡張期血圧 (mmHg) |
| SkinThickness | 皮下脂肪厚 (mm) |
| Insulin | 血中インスリン (μU/ml) |
| BMI | 体格指数 |
| DiabetesPedigreeFunction | 遺伝的リスクスコア |
| Age | 年齢 |
| **Outcome** | **糖尿病かどうか（1:あり, 0:なし）← これを予測したい！** |

In [ ]:
# インターネット上からデータを読み込む
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
df = pd.read_csv(url)

# データの大きさと最初の5行を確認
print("データの形（行数, 列数）:", df.shape)
df.head()

In [ ]:
# 糖尿病の人はどれくらいいる？
print("糖尿病の人数:")
print(df['Outcome'].value_counts())
print(f"\n糖尿病の割合: {df['Outcome'].mean()*100:.1f}%")

# 基本的な統計情報
df.describe()

### 上の表をよく見てみましょう

`min`（最小値）の行に注目してください。血糖値（Glucose）やBMIの最小値が **0** になっています。

**血糖値が0の人は生きていられません。** これは「測定されなかった」データが0として記録されてしまっているのです。実際の医療データにはこういう「きたない」部分がよくあります。

## 3. データのそうじ（10分）

ありえない0の値をそのまま使うと、モデルが間違ったパターンを学んでしまいます。
今回は、0を「その列の真ん中の値（中央値）」で置き換えるという、シンプルで実用的な方法を使います。

In [ ]:
# 0がありえない列
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# それぞれの列に0が何件あるか確認
print("0の件数:")
for col in zero_cols:
    print(f"  {col}: {(df[col] == 0).sum()}件")

In [ ]:
# 0を中央値で置き換える
df_clean = df.copy()  # 元データは残しておく

for col in zero_cols:
    median_val = df_clean[df_clean[col] > 0][col].median()  # 0以外の中央値
    df_clean.loc[df_clean[col] == 0, col] = median_val

# 確認：もう0はないはず
print("そうじ後の0の件数:")
for col in zero_cols:
    print(f"  {col}: {(df_clean[col] == 0).sum()}件")

print("\nデータのそうじが完了しました！")

In [ ]:
# そうじ後のデータで、糖尿病の有無による違いをグラフで見てみる
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['Glucose', 'BMI']):
    df_clean[df_clean['Outcome']==0][col].hist(bins=30, alpha=0.5, label='糖尿病なし', ax=ax, color='blue')
    df_clean[df_clean['Outcome']==1][col].hist(bins=30, alpha=0.5, label='糖尿病あり', ax=ax, color='red')
    ax.set_xlabel(col)
    ax.set_ylabel('人数')
    ax.set_title(f'{col}の分布')
    ax.legend()

plt.tight_layout()
plt.show()

グラフから、糖尿病の人は血糖値もBMIも高い傾向があることが分かります。
この「傾向」をコンピュータに学ばせるのが機械学習です。

## 4. 予測モデルを作る（10分）

### 大事な考え方：データを2つに分ける

- **学習用データ（8割）**: モデルに学ばせるためのデータ
- **テスト用データ（2割）**: 学習には使わず、答え合わせ専用にとっておくデータ

テストの問題を事前に知っていたら実力は測れませんよね。それと同じで、
モデルの本当の実力は「見たことのないデータ」で測る必要があります。

In [ ]:
# 説明変数（ヒントになる情報）と目的変数（当てたいもの）に分ける
X = df_clean.drop('Outcome', axis=1)  # Outcome以外の全列
y = df_clean['Outcome']               # 当てたいもの

# 学習用8割・テスト用2割に分ける
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"学習用データ: {len(X_train)}人分")
print(f"テスト用データ: {len(X_test)}人分")

In [ ]:
# ロジスティック回帰というモデルを作って学習させる
# （検査値から「糖尿病である確率」を計算するモデル。医学研究の定番です）
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)  # ← この1行が「学習」です！

print("学習が完了しました！")

## 5. どれくらい当たるか確かめる（5分）

とっておいたテスト用データ（モデルが見たことのない154人分）で答え合わせをします。

In [ ]:
# テスト用データで予測して、正解と比べる
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"正解率: {accuracy*100:.1f}%")

In [ ]:
# 予測の内訳を表で確認（混同行列といいます）
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['糖尿病なし', '糖尿病あり'],
            yticklabels=['糖尿病なし', '糖尿病あり'])
plt.ylabel('実際')
plt.xlabel('モデルの予測')
plt.title('予測の内訳')
plt.show()

print("読み方：左上と右下が「正解」、右上と左下が「間違い」です。")
print("医療では特に「実際は糖尿病なのに、なしと予測（見逃し）」に注意が必要です。")

### おまけ：目に見えるモデル「決定木」

もう1つの人気モデル「決定木」も試してみましょう。
「もし血糖値が◯◯以上なら…」という質問を繰り返して判定する、診断フローチャートのようなモデルです。

In [ ]:
# 決定木モデルを作って学習
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

tree_accuracy = accuracy_score(y_test, tree_model.predict(X_test))
print(f"決定木の正解率: {tree_accuracy*100:.1f}%")

# 木の中身を図にする
plt.figure(figsize=(18, 8))
plot_tree(tree_model, feature_names=list(X.columns),
          class_names=['糖尿病なし', '糖尿病あり'],
          filled=True, rounded=True, fontsize=10)
plt.title('決定木の中身（上から順に質問に答えていくイメージ）')
plt.show()

## 6. 新しい患者さんで予測してみる（5分）

作ったモデルを実際に使ってみましょう。数値を自由に変えて、リスクがどう変わるか試してみてください。

In [ ]:
# 新しい患者さんのデータ（数値を変えて実験してみよう！）
new_patient = pd.DataFrame([{
    'Pregnancies': 2,       # 妊娠回数
    'Glucose': 145,         # 血糖値
    'BloodPressure': 75,    # 血圧
    'SkinThickness': 30,    # 皮下脂肪厚
    'Insulin': 120,         # インスリン
    'BMI': 31.5,            # BMI
    'DiabetesPedigreeFunction': 0.5,  # 遺伝的リスク
    'Age': 48               # 年齢
}])

# 糖尿病である確率を予測
probability = model.predict_proba(new_patient)[0][1]

print(f"この患者さんが糖尿病である確率: {probability*100:.1f}%")

if probability >= 0.5:
    print("→ 高リスク：精密検査をおすすめします")
else:
    print("→ 低リスク：定期的な健診を続けましょう")

## まとめ

45分で、実際の医療データから予測モデルを作るところまでやり切りました！

### 今日学んだ流れ
1. **データを読む・眺める** — 分析はデータをよく見ることから
2. **データのそうじ** — ありえない値（0）を中央値で置き換え
3. **学習用とテスト用に分ける** — 本当の実力は「見たことのないデータ」で測る
4. **モデルの学習** — `model.fit()` の1行でコンピュータがパターンを学ぶ
5. **評価** — 正解率と予測の内訳（見逃しに注意）
6. **予測** — 新しいデータに対してモデルを使う

### 大事な心構え
- モデルの予測は**あくまで参考情報**。最終判断は人間（医師）が行います
- モデルの質はデータの質で決まります。「きれいな」データ収集が何より大切です

### 次のステップ
- `Statistics_basics.ipynb` — 同じデータで統計分析（相関・検定）を学ぶ
- `Machine_learning.ipynb` — 今日の内容の詳しい版（欠損値処理の比較、ROC曲線など）
- 新しい患者さんの数値をいろいろ変えて、モデルの反応を観察してみる

**おつかれさまでした！**

## おまけ：AIと一緒に学ぶ（時間が余ったら／自宅で）

Google Colabに入っているAIアシスタント（Gemini、画面右上の✦マーク）は、機械学習の勉強でも強力な相棒になります。無料で使えます。

### 体験1：モデルの改善をAIに相談する

Geminiのチャット欄に、こう聞いてみましょう：

> **決定木の max_depth を3から5に変えると何が起きますか？試すコードも書いて**

出てきたコードを新しいセルで実行して、正解率がどう変わるか確かめてみてください。
「深くすれば良くなる」とは限らないことが体感できるはずです（過学習といいます）。

### 体験2：結果の解釈をAIに聞く

> **混同行列で「実際は糖尿病なのに、なしと予測」が多いとき、医療現場では何が問題になりますか？**

コードを書くだけでなく、**結果の意味を考える壁打ち相手**としてもAIは役立ちます。
ただし医学的な判断は必ず自分の知識と文献で裏を取りましょう。

### 大事な心がまえ

- **AIの答えは間違うこともあります**。実行して確かめる・文献で裏を取るのは自分の仕事です
- **患者さんの実データをAIチャットに貼り付けない**こと（今日の講座データは公開研究用データなので練習に使えます）

自宅で続けたい人は、リポジトリの `AI_study_guide.md`（AIと一緒に学ぶガイド）を見てください。